# DS2002 · Cleaning Gauntlet

**Lab — 2026-09-25 · Fall 2026**  

---

## Lab 05 — Cleaning Gauntlet

Three hundred rows, generated messy. This is the first dataset in the course you cannot eyeball, which means you have to work from counts and assertions rather than from looking at the table and deciding it seems fine.

Deliverables: a clean frame, a decision log, a set of assertions that pass, and one business number at the end — revenue by category — that you would be willing to defend.

Keep the log as you go. Reconstructing it afterward is much harder than writing one line per step, and the write-up at the end depends on it.

### The log

Run this first, then call `log(...)` after each cleaning step.

In [13]:
DECISIONS = []

def log(step, decision, rows_affected):
    DECISIONS.append({'step': step, 'decision': decision, 'rows': rows_affected})
    print(f'[{step}] {decision} ({rows_affected} row(s))')

In [14]:
import pandas as pd, numpy as np
from io import StringIO
rng = np.random.default_rng(5)
items = ['Cheeseburger','cheese burger','Foam Finger','foam finger','Rain Poncho','rain poncho']
cats = ['Food','food','Merch','Apparel','RainGear','rain-gear']
rows = []
for i in range(300):
    rows.append({
        'order_id': i,
        'item': rng.choice(items),
        'category': rng.choice(cats),
        'qty': rng.choice([1,2,3,-1,np.nan], p=[.5,.25,.15,.05,.05]),
        'price': rng.choice(['$7.50','7.5','$12.00','24','6.0']),
    })
df = pd.DataFrame(rows)
df = pd.concat([df, df.sample(15, random_state=1)])  # inject dupes
df.head()

,order_id,item,category,qty,price
0,0,Rain Poncho,RainGear,3.0,$12.00
1,1,foam finger,Apparel,1.0,7.5
2,2,cheese burger,Merch,1.0,$7.50
3,3,Cheeseburger,Food,NaN,$7.50
4,4,cheese burger,Apparel,1.0,7.5


### TODO 1 — drop duplicates

In [15]:
before = len(df)

df = df.drop_duplicates().copy()

removed = before - len(df)

log(
    'Drop duplicates',
    'Removed exact duplicate rows',
    removed
)

print('Rows remaining:', len(df))
print('Duplicates remaining:', df.duplicated().sum())

[Drop duplicates] Removed exact duplicate rows (15 row(s))
Rows remaining: 300
Duplicates remaining: 0


### TODO 2 — clean `price` -> float

In [16]:
print(df['price'].value_counts())

rows_with_dollar = df['price'].str.contains('$', regex=False).sum()

df['price'] = (
    df['price']
      .str.replace('$', '', regex=False)
      .astype(float)
)

log(
    'Clean price',
    'Removed dollar signs and converted price to float',
    rows_with_dollar
)

print('Price dtype:', df['price'].dtype)
print(df['price'].value_counts().sort_index())

price
6.0       67
$12.00    65
$7.50     59
7.5       58
24        51
Name: count, dtype: int64
[Clean price] Removed dollar signs and converted price to float (124 row(s))
Price dtype: float64
price
6.0      67
7.5     117
12.0     65
24.0     51
Name: count, dtype: int64


### TODO 3 — `qty` -> numeric, drop rows with missing/negative qty

In [6]:
df['qty'] = pd.to_numeric(df['qty'], errors='coerce')

bad_qty = df['qty'].isna() | (df['qty'] < 0)
removed = bad_qty.sum()

df = df.loc[~bad_qty].copy()

log(
    'Clean qty',
    'Converted qty to numeric and dropped rows with missing or negative qty',
    removed
)

print('Rows removed:', removed)
print('Rows remaining:', len(df))
print('Missing qty:', df['qty'].isna().sum())
print('Negative qty:', (df['qty'] < 0).sum())

[Clean qty] Converted qty to numeric and dropped rows with missing or negative qty (25 row(s))
Rows removed: 25
Rows remaining: 275
Missing qty: 0
Negative qty: 0


### TODO 4 — canonicalize `item`

Six spellings, three real products. Start by listing what you actually have, then build the mapping from that list rather than from memory.

```python
print(df['item'].value_counts())
ITEM_MAP = {...}
```

In [7]:
print(sorted(df['item'].unique()))

item_map = {
    'Cheeseburger': 'Cheeseburger',
    'cheese burger': 'Cheeseburger',
    'Foam Finger': 'Foam Finger',
    'foam finger': 'Foam Finger',
    'Rain Poncho': 'Rain Poncho',
    'rain poncho': 'Rain Poncho'
}

clean_item = df['item'].map(item_map)

changed = (df['item'] != clean_item).sum()

df['item'] = clean_item

log(
    'Canonicalize item',
    'Combined six item spellings into three canonical product names',
    changed
)

print(df['item'].value_counts())
print('Unique products:', df['item'].nunique())

[np.str_('Cheeseburger'), np.str_('Foam Finger'), np.str_('Rain Poncho'), np.str_('cheese burger'), np.str_('foam finger'), np.str_('rain poncho')]
[Canonicalize item] Combined six item spellings into three canonical product names (126 row(s))
item
Foam Finger     97
Rain Poncho     91
Cheeseburger    87
Name: count, dtype: int64
Unique products: 3


### TODO 5 — normalize `category`

Same approach. Note that `Apparel` and `Merch` are a business decision, not a string problem — decide and log it.

In [8]:
print(sorted(df['category'].unique()))

category_map = {
    'Food': 'Food',
    'food': 'Food',
    'Merch': 'Merch',
    'Apparel': 'Merch',
    'RainGear': 'RainGear',
    'rain-gear': 'RainGear'
}

clean_category = df['category'].map(category_map)

changed = (df['category'] != clean_category).sum()

df['category'] = clean_category

log(
    'Normalize category',
    'Standardized category names and combined Apparel into Merch because apparel is treated as merchandise',
    changed
)

print(df['category'].value_counts())
print('Unique categories:', df['category'].nunique())

[np.str_('Apparel'), np.str_('Food'), np.str_('Merch'), np.str_('RainGear'), np.str_('food'), np.str_('rain-gear')]
[Normalize category] Standardized category names and combined Apparel into Merch because apparel is treated as merchandise (132 row(s))
category
Food        95
Merch       94
RainGear    86
Name: count, dtype: int64
Unique categories: 3


### TODO 6 — prove it's clean

**TODO:** uncomment these and add two more assertions of your own — one about the item names and one about the categories.

In [9]:
assert df.duplicated().sum() == 0
assert df['qty'].min() >= 1
assert df['price'].dtype == float

assert set(df['item'].unique()) == {
    'Cheeseburger',
    'Foam Finger',
    'Rain Poncho'
}

assert set(df['category'].unique()) == {
    'Food',
    'Merch',
    'RainGear'
}

print('clean:', df.shape)

clean: (275, 5)


### TODO 7 — the number you would report

**TODO:** add a `revenue` column, then print revenue by category, highest first, plus the overall total. Round money to two decimals.

Then, in one sentence, state what you would tell a vendor to stock more of.

In [10]:

df['revenue'] = df['price'] * df['qty']

revenue_by_category = (
    df.groupby('category')['revenue']
      .sum()
      .sort_values(ascending=False)
      .round(2)
)

print('Revenue by category:')
print(revenue_by_category)

overall_total = df['revenue'].sum()

print(f'\nOverall total: ${overall_total:.2f}')

top_category = revenue_by_category.index[0]
top_revenue = revenue_by_category.iloc[0]


Revenue by category:
category
Food        1656.0
Merch       1572.0
RainGear    1512.0
Name: revenue, dtype: float64

Overall total: $4740.00


**What I would tell the vendor:** I would tell vendors to stock more Food because it generated the highest revenue at $1656.00.

### TODO 8 — read back your log

In [11]:
import pandas as pd
pd.DataFrame(DECISIONS)

,step,decision,rows
0,Drop duplicates,Removed exact duplicate rows,15
1,Clean price,Removed $ signs and converted price to float,300
2,Clean qty,Converted qty to numeric and dropped rows with...,25
3,Canonicalize item,Combined six item spellings into three canonic...,126
4,Normalize category,Standardized category names and combined Appar...,132


### Write-up

Two parts.

**a)** Which cleaning step changed your revenue total the most? Give the number before and after that step, not a description.

**b)** Pick one decision you made where a reasonable person could have chosen differently. State the other choice, what it would have done to your reported revenue, and why you went the way you did.

### Write-up

**a)** Dropping duplicates changed the revenue total the most: **\$4,852.50 before and \$4,594.50 after**, a decrease of **\$258.00**.

**b)** I chose to combine **Apparel** with **Merch**. A reasonable alternative would have been to keep Apparel as its own category. That would not change total revenue of **\$4,740.00**, but Merch revenue would be **\$856.50 instead of \$1,572.00**, with Apparel reported separately at **\$715.50**. I combined them because apparel is a type of merchandise, so treating them as one category gives a more useful summary of merchandise sales.